# Phụ lục: Bằng chứng và tái lập

## Nguồn kết quả

TV3 và TV4 đã bổ sung audit ngày 04/11; bộ ZIP sửa được nhập vào snapshot `updated-20261106`. Audit TV4 được chọn từ bản có thời điểm tạo mới nhất trong hai ZIP được gửi. Cả hai bản đều ghi 5.000 record, 0 unseen/unresolved, 4.882 pass và 118 fail. Bản TV2/TV5 và mọi snapshot cũ được giữ nguyên.

Ngày cập nhật báo cáo là 06/11/2026; ngày kiểm tra thực tế nằm trong từng audit. Kiểm hash artifacts khi build bảo đảm đúng tệp đã lưu, chưa thay việc tái kiểm corpus sống. ZIP TV5 vẫn là bản cục bộ cùng tên/kích thước bản chia sẻ, chưa đối chiếu hash cloud.

Thông tin nhóm đã chạy 5.000 ID/người được nhóm cung cấp; số kiểm chứng trong bảng lấy từ audit. Thư mục nguồn: [arxiv_ms1_data](https://drive.google.com/drive/folders/123TUmjXMX38gKRLnJIKDl2yZF8jK6e9n).

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
sys.path.insert(0, str(ROOT / 'utils'))
from preliminary_results import (load_members, progress_rows, assignment_outcomes,
    quality_rows, error_groups, performance_rows, run_range, session_members, coverage_rows, cumulative_rows)
members = load_members(ROOT)
sessions = session_members(ROOT, members)
def table(rows):
    frame = pd.DataFrame(rows, dtype=object)
    frame = frame.where(frame.notna(), 'Chưa đủ bằng chứng')
    def number(value):
        if isinstance(value, float):
            return f'{value:,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
        if isinstance(value, int) and not isinstance(value, bool):
            return f'{value:,}'.replace(',', '.')
        return str(value)
    display(HTML(frame.map(number).to_html(index=False, escape=True, border=0, classes='dataframe')))


In [ ]:
for s in sessions:
    p=s['provenance']
    display(Markdown('**'+s['session']+'**: ['+p['archive_name']+']('+p['drive_url']+')'))
    display(Markdown('SHA-256 ZIP: `'+p['archive_sha256']+'`'))
table([{'Thành viên':m['member'],'Audit tạo lúc (UTC)':m['audit']['generated_at'] if m['audit'] else None,
        'Nguồn':m['provenance']['drive_url'] if m['provenance'] else None} for m in members])
table([{'Phiên':s['session'],'Run ID':s['benchmark']['run_id'],'Bắt đầu (UTC)':s['benchmark']['started_at'],
        'Kết thúc (UTC)':s['benchmark']['finished_at'],'Source commit':s['benchmark']['environment'].get('source_commit'),
        'CPU logic':s['benchmark']['environment'].get('logical_cpu_count'),
        'CPU-only ghi nhận':s['benchmark']['environment'].get('cpu_only_confirmation')} for s in sessions])

## Đối chiếu audit

Ở các commit source mà benchmark khai báo, cờ hoàn tất audit yêu cầu không còn ID chưa thấy, Discovery chưa giải quyết, validation fail và diagnostics. Vì vậy `audit_complete=false` của TV3/TV4 đến từ validation, trong khi cả hai đã có đủ record và lookup được giải quyết.

Phạm vi TV3/TV4/TV5 được đối chiếu với input đi cùng bản audit trùng nội dung; TV2 liệt kê đủ 5.000 định danh đúng dải. Trường `discovered_assigned_ids` đếm ID có record, kể cả record lỗi. Số Discovery thành công được đọc riêng từ bảng trạng thái.

Audit TV2 ghi `audit_complete=true` cùng 2.579 paper fail, chưa phù hợp công thức source được khai báo. Cần xác minh code audit thực dùng và độ mới của validation. TV5 còn bốn Discovery lỗi: 2410.04689, 2410.04690, 2410.05043, 2410.05044.

In [ ]:
table(coverage_rows(members))

## Xuất thống kê corpus tích lũy

Notebook [MS1_Cumulative_Statistics.ipynb](https://colab.research.google.com/github/HoangLongNha/arxiv-project-report/blob/main/colab/MS1_Cumulative_Statistics.ipynb) đọc dữ liệu đang có trên Drive. Dừng các tiến trình ghi trước khi chạy, chọn đúng thành viên và đường dẫn `data/`. Notebook chỉ xuất một tệp JSON mới ở ngoài corpus; không gọi API thu thập và không sửa checkpoint/corpus.

Công cụ đọc mọi checkpoint Discovery trên dải được giao, đối chiếu stage/tệp và tránh đếm lặp paper/version qua nhiều phiên. References được đọc một lần mỗi paper. Trước/sau lọc chỉ tính trên những paper có đủ sidecar; số paper tạo mẫu số được ghi trong kết quả. Validation được đọc từ báo cáo đã có, không được coi là một lượt revalidate mới.

Kết quả `cumulative-statistics.json` cần được gửi lại cho TV5 để đối chiếu/import. Không đổi run ID của statistics resume thành toàn dải và không ghi đè manifest để ép tạo thống kê. Khi đủ tệp của năm thành viên mới tổng hợp số liệu toàn nhóm.

## Vai trò từng tệp

| Tệp | Phạm vi |
|---|---|
| assignment-audit.json | Coverage và validation trên dải audit |
| inputs.json | Dải dự định kiểm tra, chưa phải kết quả thực thi |
| statistics.json | Các paper được manifest của lượt chạy chọn |
| cumulative-statistics.json | Toàn dải từ checkpoint và dữ liệu tích lũy, có phạm vi riêng |
| benchmark/samples/timings | Thời gian và tài nguyên của một phiên |
| provenance.json | Nguồn ZIP và hash gốc/bản công bố |

README hướng dẫn cài môi trường, export và build. Notebook báo cáo vẫn thực thi khi build; code tính toán được ẩn khỏi HTML. Tài liệu bàn giao gốc nằm ở [Đóng góp thành viên](member-contributions.md).